<a href="https://colab.research.google.com/github/Abhishek2993/BrainCancer-Detection-MRI-DATA-/blob/main/Phase1_Rigour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BCDC Phase 1: Rigour

Phase 0 made the pipeline correct. Phase 1 asks whether the model can be **trusted**:

| Section | Question |
|---|---|
| **A. Deep ensemble** | Does combining the Phase 0 seed models help, and can the ensemble tell when it's unsure? |
| **B. Conformal prediction** | Can we output a *set* of labels that contains the true class at least 90% (or 95%) of the time, for each tumour type? |
| **C. External validation** | How much does performance drop on a **different dataset** (Figshare, Cheng et al.: other hospitals, other scanners, patient IDs available)? |
| **D. Out-of-distribution rejection** | Can the model say *"this doesn't look like the MRIs I was trained on"* instead of guessing? |
| **E. Verified explainability** | Do Grad-CAM heatmaps actually fall on the tumour? Measured against radiologist-drawn masks and compared with trivial baselines. |

**Before running:** run the Phase 0 notebook (`Project_BCDCipynb.ipynb`) in Colab first. It saves the trained seed models to Google Drive (`MyDrive/BCDC/outputs`), and this notebook loads them from there.

Use a GPU runtime (*Runtime → Change runtime type → T4 GPU*). The Figshare download is about 880 MB, and a full run takes roughly 20–40 minutes.

> ⚠️ Research and educational use only. Not a medical device.

## 0. Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Abhishek2993/BrainCancer-Detection-MRI-DATA-.git"
REPO_DIR = Path(os.environ.get("BCDC_REPO_DIR", "BrainCancer-Detection-MRI-DATA-"))

# The shared code lives in the repo's `bcdc` package; clone the repo if it isn't importable yet.
try:
    import bcdc
except ImportError:
    if not (REPO_DIR / "bcdc").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    sys.path.insert(0, str(REPO_DIR.resolve()))
    import bcdc
print("bcdc", bcdc.__version__, "from", Path(bcdc.__file__).parent)

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.metrics import roc_auc_score
from IPython.display import Markdown, display

import tensorflow as tf
from tensorflow import keras

from bcdc import figshare
from bcdc.conformal import ConformalClassifier, set_metrics
from bcdc.data import (array_dataset, best_thumbnail_match, cross_near_duplicates, dihedral_hashes,
                       display_name, find_class_root, load_split, path_dataset, query_hash, thumbnail,
                       thumbnail_variants)
from bcdc.ensemble import ensemble_probs, uncertainty
from bcdc.gradcam import centre_prior, gradcam, localisation_metrics, random_map
from bcdc.metrics import apply_temperature, evaluate, expected_calibration_error, fit_temperature, risk_coverage
from bcdc.model import feature_extractor, feature_model, model_parts, softmax
from bcdc.ood import Mahalanobis, energy_score, msp_score, ood_metrics, threshold_at_retention

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ---- Configuration -------------------------------------------------------
ALPHAS = [0.10, 0.05]          # conformal miscoverage levels (90% and 95% coverage)
DEPLOY_ALPHA = 0.10            # level used by predict_image_v2
OOD_RETAIN = 0.95              # OOD threshold keeps 95% of in-distribution validation images
N_OOD = 1000                   # images per natural-image OOD set
GRADCAM_MAX = 600              # Figshare images used for the Grad-CAM evaluation
FIGSHARE_SIZE = 256            # Figshare slices and masks are stored at this resolution
DUP_HAMMING = 4                # same near-duplicate threshold as Phase 0
DUP_MIN_CORR = 0.99            # thumbnail correlation above which two images count as copies
SEED = 0
BATCH_SIZE = 32

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if os.environ.get("BCDC_ARTEFACT_DIR"):
    ARTEFACT_DIR = Path(os.environ["BCDC_ARTEFACT_DIR"])
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTEFACT_DIR = Path("/content/drive/MyDrive/BCDC/outputs")
else:
    ARTEFACT_DIR = Path("outputs")

P1_DIR = ARTEFACT_DIR / "phase1"
FIG_DIR = P1_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

meta = json.loads((ARTEFACT_DIR / "model_meta.json").read_text())
CLASS_NAMES = meta["class_names"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = tuple(meta["img_size"])
MODEL_FILES = sorted((ARTEFACT_DIR / "models").glob("seed_*.keras"))
assert MODEL_FILES, (f"No seed models in {ARTEFACT_DIR / 'models'}. Run the Phase 0 notebook first "
                     "(with PERSIST_TO_DRIVE = True in Colab).")
NAMES = [display_name(c) for c in CLASS_NAMES]
print("Artefacts:", ARTEFACT_DIR, "| classes:", CLASS_NAMES, "| image size:", IMG_SIZE,
      "| seed models:", [f.stem for f in MODEL_FILES])


def md_table(df, floatfmt="{:.3f}"):
    # Small Markdown table helper (avoids an extra dependency).
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join([""] + cols) + " |", "|" + "---|" * (len(cols) + 1)]
    for idx, row in df.iterrows():
        vals = [floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in row]
        lines.append("| " + " | ".join([str(idx)] + vals) + " |")
    return "\n".join(lines)


REPORT = []   # Markdown sections collected for results_phase1.md

## 1. Internal data (the Phase 0 split)

We reuse Phase 0's train / validation / test split exactly. The validation set is divided again, by duplicate group, into two halves:

* **val-A** fits the ensemble temperature and the OOD threshold;
* **val-B** calibrates the conformal prediction sets.

Keeping them separate matters. Conformal guarantees only hold when the calibration data wasn't also used to tune the scores being calibrated.

In [ ]:
if os.environ.get("BCDC_DATASET_PATH"):
    DATASET_PATH = find_class_root(os.environ["BCDC_DATASET_PATH"])
else:
    import kagglehub
    path = kagglehub.dataset_download("orvile/brain-cancer-mri-dataset")
    expected = Path(path) / "Brain_Cancer raw MRI data" / "Brain_Cancer"
    DATASET_PATH = expected if expected.is_dir() else find_class_root(path)

split = load_split(ARTEFACT_DIR / "split.csv", DATASET_PATH)
val_groups = np.array(sorted(split.loc[split["split"] == "val", "group"].unique()))
np.random.default_rng(SEED).shuffle(val_groups)
half_a = set(val_groups[: len(val_groups) // 2])
is_val = split["split"] == "val"
split.loc[is_val & split["group"].isin(half_a), "split"] = "valA"
split.loc[is_val & ~split["group"].isin(half_a), "split"] = "valB"

SPLITS = ["train", "valA", "valB", "test"]
Y = {s: split.loc[split["split"] == s, "label"].to_numpy() for s in SPLITS}
PATHS = {s: split.loc[split["split"] == s, "abs_path"].to_numpy() for s in SPLITS}
print(pd.crosstab(split["cls"], split["split"])[SPLITS])

## 2. Load the seed models and extract features

For every model and every split we store the **pooled penultimate features** (1,280-d for MobileNetV2) and the **logits** (pre-softmax scores). Logits are needed for the energy OOD score, and features for the Mahalanobis OOD score.

In [ ]:
members = []
for f in MODEL_FILES:
    t0 = time.time()
    model = keras.models.load_model(f)
    parts = model_parts(model)
    m = dict(name=f.stem, model=model, parts=parts, fmodel=feature_model(parts),
             extractor=feature_extractor(parts), feats={}, logits={})
    for s in SPLITS:
        feats = m["fmodel"].predict(path_dataset(PATHS[s], IMG_SIZE, BATCH_SIZE), verbose=0)
        m["feats"][s], m["logits"][s] = feats, feats @ parts.W + parts.b
    # sanity check: our logits reproduce the saved model's softmax output
    check = model.predict(path_dataset(PATHS["test"][:16], IMG_SIZE, BATCH_SIZE), verbose=0)
    assert np.allclose(softmax(m["logits"]["test"][:16]), check, atol=1e-4)
    members.append(m)
    print(f"{f.stem}: features extracted in {time.time() - t0:.0f}s")


def member_probs(key):
    return np.stack([softmax(m["logits"][key]) for m in members])   # (M, N, K)

## A. Deep ensemble and uncertainty

The ensemble averages the members' probabilities. A single temperature, fitted on val-A, then calibrates the average.

Ensemble uncertainty splits into two parts:

* **aleatoric:** all members agree the case is ambiguous;
* **epistemic:** the members *disagree*. This often signals an unusual input.

**Selective classification:** if the model may decline its least-certain cases (and refer them to a human), how fast does the error rate fall? The risk–coverage curve shows this, and **AURC** (area under it) summarises it; lower is better.

In [ ]:
P = {s: member_probs(s) for s in SPLITS}
T_ENS = fit_temperature(Y["valA"], ensemble_probs(P["valA"]))


def ens_calibrated(member_p):
    return apply_temperature(ensemble_probs(member_p), T_ENS)


rows = {}
for i, m in enumerate(members):
    T_i = fit_temperature(Y["valA"], P["valA"][i])
    raw, _ = evaluate(Y["test"], P["test"][i], CLASS_NAMES)
    rows[m["name"]] = {**{k: raw[k] for k in ["accuracy", "macro_f1", "macro_auroc", "nll", "ece"]},
                       "ece_after_T": expected_calibration_error(Y["test"], apply_temperature(P["test"][i], T_i))}
raw, _ = evaluate(Y["test"], ensemble_probs(P["test"]), CLASS_NAMES)
cal, ens_cm = evaluate(Y["test"], ens_calibrated(P["test"]), CLASS_NAMES)
rows[f"ensemble ({len(members)} members)"] = {**{k: raw[k] for k in ["accuracy", "macro_f1", "macro_auroc", "nll", "ece"]},
                                             "ece_after_T": cal["ece"]}
ens_table = pd.DataFrame(rows).T
display(Markdown(f"**Internal test set ({len(Y['test'])} images)** · ensemble temperature T = {T_ENS:.2f}\n\n" + md_table(ens_table)))
REPORT.append("### A. Deep ensemble (internal test set)\n\n" + md_table(ens_table))

# Selective classification
p_test = ens_calibrated(P["test"])
correct = p_test.argmax(1) == Y["test"]
unc = uncertainty(P["test"])
confidences = {"max probability": msp_score(p_test), "-total entropy": -unc["total"],
               "-epistemic (disagreement)": -unc["epistemic"]}
plt.figure(figsize=(6, 4))
aurc = {}
for name, conf in confidences.items():
    cov, risk, aurc[name] = risk_coverage(correct, conf)
    plt.plot(cov, risk, label=f"{name} (AURC {aurc[name]:.4f})")
plt.xlabel("coverage (share of cases the model keeps)"); plt.ylabel("error rate on kept cases")
plt.title("Risk–coverage on the internal test set"); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(FIG_DIR / "risk_coverage.png", dpi=120); plt.show()
for target in (0.9, 0.8):
    k = int(np.ceil(target * len(correct)))
    kept = np.argsort(-confidences["max probability"])[:k]
    print(f"Keeping the {100 * target:.0f}% most confident cases: error {100 * (1 - correct[kept].mean()):.2f}% "
          f"(vs {100 * (1 - correct.mean()):.2f}% on all)")
REPORT.append("**Selective classification (AURC, lower is better):** " +
              ", ".join(f"{k}: {v:.4f}" for k, v in aurc.items()))

## B. Conformal prediction sets

Calibrated on **val-B** and evaluated on the internal test set.

* **LAC** gives the smallest sets on average.
* **APS** gives sets that grow when the model is unsure.
* **Class-conditional LAC** guarantees the coverage level *separately for each tumour type*. Without it, a common class can "carry" a rarer, harder one. This is the variant used for deployment.

What to look for: **coverage** should be ≥ 1 − α (e.g. ≥ 0.90), with sets as small as possible.

In [ ]:
p_cal, y_cal = ens_calibrated(P["valB"]), Y["valB"]
CP_CONFIGS = {"LAC": dict(method="lac"),
              "APS (randomised)": dict(method="aps", randomized=True, seed=SEED),
              "LAC, class-conditional": dict(method="lac", class_conditional=True)}
conf_rows = {}
CP = {}
for alpha in ALPHAS:
    for name, kw in CP_CONFIGS.items():
        cp = ConformalClassifier(alpha, **kw).fit(p_cal, y_cal)
        CP[(alpha, name)] = cp
        sm = set_metrics(cp.predict_sets(p_test), Y["test"], CLASS_NAMES)
        conf_rows[f"{name}, α={alpha}"] = {"target": 1 - alpha, **{k.replace("coverage[", "cov[").replace("]", "]"): v
                                                                   for k, v in sm.items()}}
conf_table = pd.DataFrame(conf_rows).T
conf_table.columns = [c if not c.startswith("cov[") else "cov " + display_name(c[4:-1]) for c in conf_table.columns]
display(Markdown(f"**Internal test set** · calibrated on val-B ({len(y_cal)} images)\n\n" + md_table(conf_table)))
REPORT.append("### B. Conformal prediction (internal test set)\n\n" + md_table(conf_table))

CP_DEPLOY = CP[(DEPLOY_ALPHA, "LAC, class-conditional")]
sets = CP_DEPLOY.predict_sets(p_test)
print("Examples of deployed prediction sets (class-conditional LAC, α = %.2f):" % DEPLOY_ALPHA)
for i in np.r_[np.where(sets.sum(1) > 1)[0][:3], np.where(sets.sum(1) == 1)[0][:2]]:
    print(f"  true {display_name(CLASS_NAMES[Y['test'][i]]):<32s} set {{{', '.join(NAMES[k] for k in np.where(sets[i])[0])}}}")

## C. External validation on the Figshare dataset (Cheng et al.)

[Figshare brain-tumour dataset](https://figshare.com/articles/dataset/brain_tumor_dataset/1512427): 3,064 T1-weighted contrast-enhanced slices from **233 patients**, with labels **glioma, meningioma, pituitary** and a **tumour mask** for every slice. It comes from different hospitals and scanners than the training data.

* Only **glioma** and **meningioma** exist in both datasets, so the headline external metrics use those two classes. The model can still predict its third class (`brain_tumor`), and how often it does is reported.
* **Pituitary** has no matching training class. It is used in section D as an *unseen tumour type*.
* Figshare slices that match **any** training-dataset image are **excluded**, because public brain-MRI collections often reuse each other's images. A slice counts as a match if its hash is within 4 bits or its thumbnail correlation is ≥ 0.99, across all flips and rotations. A histogram and the top-matched pairs are plotted so the threshold can be checked by eye.
* Patient IDs allow a **patient-level** result: the probabilities are averaged over each patient's slices.

> Please cite Cheng et al. (PLoS ONE 2015, 2016) if you use this dataset.

In [ ]:
FIGSHARE_DIR = Path(os.environ["BCDC_FIGSHARE_DIR"]) if os.environ.get("BCDC_FIGSHARE_DIR") else figshare.download("data/figshare")
fs_meta, fs_img, fs_mask = figshare.load(FIGSHARE_DIR, size=FIGSHARE_SIZE)
print(fs_meta["label"].value_counts().to_string(), f"\n{fs_meta['pid'].nunique()} patients")

# Cross-dataset duplicate check against every image of the training dataset (all splits).
# Two signals, because the datasets were resized/rescaled differently, which can push the
# hash of a true copy past DUP_HAMMING bits:
#   (1) dHash within DUP_HAMMING bits, or (2) 32x32 thumbnail correlation >= DUP_MIN_CORR
#   (both checked across all 8 flips/rotations).
t0 = time.time()
kaggle_hashes, kaggle_thumbs = [], []
for p in split["abs_path"]:                      # one image at a time keeps memory low
    with Image.open(p) as im:
        im_l = im.convert("L")
    kaggle_hashes.append(dihedral_hashes(im_l))
    kaggle_thumbs.append(thumbnail_variants(im_l))
kaggle_hashes, kaggle_thumbs = np.array(kaggle_hashes, dtype=np.uint64), np.stack(kaggle_thumbs)
fs_q_hash = [query_hash(Image.fromarray(im)) for im in fs_img]
fs_q_thumb = np.stack([thumbnail(Image.fromarray(im)) for im in fs_img])
hash_hit = cross_near_duplicates(fs_q_hash, kaggle_hashes, DUP_HAMMING)
best_corr, best_ref = best_thumbnail_match(fs_q_thumb, kaggle_thumbs)
corr_hit = best_corr >= DUP_MIN_CORR
fs_meta["dup_of_training_data"] = hash_hit | corr_hit
fs_meta["best_corr"], fs_meta["best_match"] = best_corr, split["path"].to_numpy()[best_ref]
print(f"Figshare slices matching a training-dataset image: {int(fs_meta['dup_of_training_data'].sum())} excluded "
      f"(hash: {int(hash_hit.sum())}, thumbnail correlation: {int(corr_hit.sum())}) [{time.time() - t0:.0f}s]")

# Check the threshold by eye: the distribution of best matches, and the top-matched pairs.
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.hist(best_corr, bins=60, color="grey"); ax.axvline(DUP_MIN_CORR, color="red", ls="--", label="threshold")
ax.set_xlabel("best thumbnail correlation with any training-dataset image"); ax.set_ylabel("Figshare slices")
ax.legend(); plt.tight_layout(); plt.savefig(FIG_DIR / "cross_dataset_match_hist.png", dpi=120); plt.show()
top = np.argsort(-best_corr)[:5]
fig, axes = plt.subplots(2, len(top), figsize=(2.3 * len(top), 4.8), squeeze=False)
for j, i in enumerate(top):
    axes[0, j].imshow(fs_img[i], cmap="gray"); axes[0, j].set_title(f"Figshare\ncorr {best_corr[i]:.3f}", fontsize=7)
    axes[1, j].imshow(Image.open(DATASET_PATH / fs_meta["best_match"][i]).convert("L"), cmap="gray")
    axes[1, j].set_title("best training-set match" + (" (DUP)" if fs_meta["dup_of_training_data"][i] else "")
                         + "\n(may be flipped/rotated)", fontsize=7)
    for a in axes[:, j]:
        a.axis("off")
plt.tight_layout(); plt.savefig(FIG_DIR / "cross_dataset_top_matches.png", dpi=120); plt.show()

fs_ds = array_dataset(fs_img, IMG_SIZE, BATCH_SIZE)
for m in members:
    feats = m["fmodel"].predict(fs_ds, verbose=0)
    m["feats"]["figshare"], m["logits"]["figshare"] = feats, feats @ m["parts"].W + m["parts"].b
P["figshare"] = member_probs("figshare")
p_fs = ens_calibrated(P["figshare"])

keep = ~fs_meta["dup_of_training_data"].to_numpy()
shared = fs_meta["label"].isin(figshare.TO_TRAINING_CLASS).to_numpy() & keep
y_fs = np.array([CLASS_NAMES.index(figshare.TO_TRAINING_CLASS.get(l, CLASS_NAMES[0])) for l in fs_meta["label"]])

In [ ]:
G, M = CLASS_NAMES.index("brain_glioma"), CLASS_NAMES.index("brain_menin")
OTHER = [k for k in range(NUM_CLASSES) if k not in (G, M)]


def two_class_report(y, p):
    # Metrics on glioma + meningioma cases only (the classes both datasets share).
    pred = p.argmax(1)
    binary = p[:, G] / np.maximum(p[:, G] + p[:, M], 1e-12)
    sets = CP_DEPLOY.predict_sets(p)
    out = {
        "n images": len(y),
        "accuracy (3-way argmax)": float((pred == y).mean()),
        "glioma sensitivity": float((pred[y == G] == G).mean()) if (y == G).any() else float("nan"),
        "meningioma sensitivity": float((pred[y == M] == M).mean()) if (y == M).any() else float("nan"),
        "predicted as 'brain_tumor'": float(np.isin(pred, OTHER).mean()),
        "glioma-vs-meningioma AUROC": float(roc_auc_score(y == G, binary)) if 0 < (y == G).sum() < len(y) else float("nan"),
        "ECE": expected_calibration_error(y, p),
        f"conformal coverage (α={DEPLOY_ALPHA})": float(sets[np.arange(len(y)), y].mean()),
        "avg set size": float(sets.sum(1).mean()),
    }
    return out


internal_gm = np.isin(Y["test"], [G, M])
gap = pd.DataFrame({
    "internal test": two_class_report(Y["test"][internal_gm], p_test[internal_gm]),
    "Figshare (external)": two_class_report(y_fs[shared], p_fs[shared]),
})
# Patient level: average probabilities over each patient's slices
pt = fs_meta[shared].assign(**{f"p{k}": p_fs[shared][:, k] for k in range(NUM_CLASSES)}).groupby("pid")
pt_probs = pt[[f"p{k}" for k in range(NUM_CLASSES)]].mean().to_numpy()
pt_y = np.array([CLASS_NAMES.index(figshare.TO_TRAINING_CLASS[l]) for l in pt["label"].first()])
gap["Figshare, patient level"] = two_class_report(pt_y, pt_probs)
display(Markdown("**Generalisation gap (glioma + meningioma only)**\n\n" + md_table(gap)))
REPORT.append("### C. External validation: Figshare (glioma + meningioma)\n\n" + md_table(gap))

# Confusion on Figshare: rows = true (glioma, meningioma), columns = predicted (all classes)
cm = np.array([[np.sum((y_fs[shared] == t) & (p_fs[shared].argmax(1) == k)) for k in range(NUM_CLASSES)] for t in (G, M)])
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.imshow(cm / cm.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{cm[i, j]}\n({cm[i, j] / max(cm[i].sum(), 1):.2f})", ha="center", va="center", fontsize=8)
ax.set_xticks(range(NUM_CLASSES), NAMES, rotation=15, fontsize=8); ax.set_yticks([0, 1], [NAMES[G], NAMES[M]], fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Figshare external test: confusion")
plt.tight_layout(); plt.savefig(FIG_DIR / "figshare_confusion.png", dpi=120); plt.show()

pit = (fs_meta["label"] == "pituitary").to_numpy() & keep
if pit.any():
    dist = pd.Series(p_fs[pit].argmax(1)).map(lambda k: NAMES[k]).value_counts(normalize=True)
    print("Pituitary slices (no matching training class) are predicted as:\n" + dist.round(3).to_string())
    REPORT.append("Pituitary slices (unseen class) predicted as: " +
                  ", ".join(f"{k} {100 * v:.1f}%" for k, v in dist.items()))

## D. Out-of-distribution rejection

We compare four scores (higher = "looks like the training data"):

* **MSP:** the ensemble's maximum probability;
* **Energy:** log-sum-exp of the logits, averaged over members;
* **−Epistemic:** minus the members' disagreement;
* **Mahalanobis:** minus the distance from an image's features to the nearest class centre in the training data, averaged over members.

OOD test sets:

* natural photos (CIFAR-10) in **colour** and in **grayscale** (the Phase 0 colour check can't catch grayscale);
* **Gaussian noise**;
* **Figshare pituitary**, an unseen tumour type (hard, "near" OOD).

Each threshold is set on **val-A** so that it keeps 95% of in-distribution images. We report AUROC, FPR at 95% TPR (the share of OOD images wrongly accepted), and the acceptance rate at that threshold.

Figshare **glioma / meningioma** is listed separately. Those images *should* be accepted, so a high rejection rate there means the rejector also refuses legitimate scans from new hospitals.

**Mahalanobis is used for deployment.** It was chosen in advance, as the standard feature-space detector, not picked after seeing the test results.

In [ ]:
rng = np.random.default_rng(SEED)
OOD_SETS = {}
if os.environ.get("BCDC_SYNTHETIC_OOD") == "1":           # offline smoke tests only
    OOD_SETS["colour photos (synthetic)"] = rng.integers(0, 255, (200, 64, 64, 3), dtype=np.uint8)
    OOD_SETS["grayscale photos (synthetic)"] = np.repeat(rng.integers(0, 255, (200, 64, 64, 1), dtype=np.uint8), 3, -1)
else:
    (_, _), (x_cifar, _) = keras.datasets.cifar10.load_data()
    pick = x_cifar[rng.choice(len(x_cifar), N_OOD, replace=False)]
    OOD_SETS["CIFAR-10 (colour)"] = pick
    gray = np.asarray([np.asarray(Image.fromarray(im).convert("L")) for im in pick])
    OOD_SETS["CIFAR-10 (grayscale)"] = np.repeat(gray[..., None], 3, -1)
OOD_SETS["Gaussian noise"] = np.clip(rng.normal(128, 50, (300, 64, 64)), 0, 255).astype(np.uint8)
if pit.any():
    OOD_SETS["Figshare pituitary (unseen tumour type)"] = fs_img[pit]

for name, arr in OOD_SETS.items():
    ds = array_dataset(arr, IMG_SIZE, BATCH_SIZE)
    for m in members:
        feats = m["fmodel"].predict(ds, verbose=0)
        m["feats"][name], m["logits"][name] = feats, feats @ m["parts"].W + m["parts"].b

MAHA = [Mahalanobis().fit(m["feats"]["train"], Y["train"]) for m in members]


def ood_scores(key, idx=None):
    sel = (lambda a: a) if idx is None else (lambda a: a[idx])
    mp = np.stack([softmax(sel(m["logits"][key])) for m in members])
    return {
        "MSP": msp_score(apply_temperature(ensemble_probs(mp), T_ENS)),
        "Energy": np.mean([energy_score(sel(m["logits"][key])) for m in members], axis=0),
        "-Epistemic": -uncertainty(mp)["epistemic"],
        "Mahalanobis": np.mean([mh.score(sel(m["feats"][key])) for mh, m in zip(MAHA, members)], axis=0),
    }


S_VAL, S_TEST = ood_scores("valA"), ood_scores("test")
THRESH = {k: threshold_at_retention(v, OOD_RETAIN) for k, v in S_VAL.items()}
rows = []
for name in OOD_SETS:
    s_ood = ood_scores(name)
    for score in S_TEST:
        r = ood_metrics(S_TEST[score], s_ood[score], THRESH[score])
        rows.append({"OOD set": name, "score": score, "AUROC": r["auroc"], "FPR@95TPR": r["fpr_at_95tpr"],
                     "OOD accepted": r["ood_accepted"]})
s_shift = ood_scores("figshare", np.where(shared)[0])
for score in S_TEST:
    rows.append({"OOD set": "Figshare glioma+meningioma (should be ACCEPTED)", "score": score,
                 "AUROC": float("nan"), "FPR@95TPR": float("nan"),
                 "OOD accepted": float((s_shift[score] >= THRESH[score]).mean())})
ood_table = pd.DataFrame(rows).set_index(["OOD set", "score"])
ood_flat = ood_table.copy()
ood_flat.index = [f"{a} · {b}" for a, b in ood_flat.index]
id_rejected = {k: float((S_TEST[k] < THRESH[k]).mean()) for k in S_TEST}
display(Markdown("**OOD detection** (ID = internal test set; thresholds keep 95% of val-A)\n\n" + md_table(ood_flat)
                 + "\n\nInternal test images wrongly rejected: "
                 + ", ".join(f"{k} {100 * v:.1f}%" for k, v in id_rejected.items())))
REPORT.append("### D. Out-of-distribution rejection\n\n" + md_table(ood_flat))

## E. Verified explainability: Grad-CAM against tumour masks

For Figshare glioma and meningioma slices we compute Grad-CAM for the ensemble's predicted class, averaged over members, and compare it with the radiologist's tumour mask:

* **Pointing game:** is the heatmap's peak on the tumour (within 15 px at 512 px)?
* **Energy in mask:** what share of the heatmap lies on the tumour?
* **IoU (area-matched):** the overlap between the tumour and the same number of hottest heatmap pixels;
* **Border energy:** the share of the heatmap in the outer 10% ring, where image borders and text sit. Lower is better.

**Baselines:** a centred Gaussian blob and random noise. A heatmap that doesn't clearly beat the *centre prior* tells us nothing about the tumour. The chance level for *energy in mask* is the tumour's share of the image.

In [ ]:
cand = np.where(shared)[0]
sel = np.sort(rng.choice(cand, min(GRADCAM_MAX, len(cand)), replace=False))
pred_fs = p_fs.argmax(1)
centre = centre_prior((FIGSHARE_SIZE, FIGSHARE_SIZE))
cam_rows, saved_cams = [], {}
for start in range(0, len(sel), BATCH_SIZE):
    b = sel[start:start + BATCH_SIZE]
    x = tf.round(tf.image.resize(np.repeat(fs_img[b][..., None], 3, -1).astype("float32"), IMG_SIZE))
    cams = np.mean([gradcam(m["extractor"], m["parts"], x, pred_fs[b], out_size=(FIGSHARE_SIZE, FIGSHARE_SIZE))
                    for m in members], axis=0)
    for i, c in zip(b, cams):
        mask = fs_mask[i]
        row = {"idx": i, "correct": bool(pred_fs[i] == y_fs[i]), "mask_fraction": float(mask.mean())}
        row.update({f"gradcam {k}": v for k, v in localisation_metrics(c, mask).items()})
        row.update({f"centre prior {k}": v for k, v in localisation_metrics(centre, mask).items()})
        row.update({f"random {k}": v for k, v in localisation_metrics(random_map(mask.shape, rng), mask).items()})
        cam_rows.append(row)
        if len(saved_cams) < 12:
            saved_cams[i] = c
cam_df = pd.DataFrame(cam_rows)
cam_df.to_csv(P1_DIR / "gradcam_per_image.csv", index=False)

metrics = ["pointing_game", "energy_in_mask", "iou_area_matched", "border_energy"]
summary = pd.DataFrame({
    "Grad-CAM (all)": [cam_df[f"gradcam {k}"].mean() for k in metrics],
    "Grad-CAM (correct preds)": [cam_df.loc[cam_df.correct, f"gradcam {k}"].mean() for k in metrics],
    "Grad-CAM (wrong preds)": [cam_df.loc[~cam_df.correct, f"gradcam {k}"].mean() for k in metrics],
    "centre-prior baseline": [cam_df[f"centre prior {k}"].mean() for k in metrics],
    "random baseline": [cam_df[f"random {k}"].mean() for k in metrics],
}, index=metrics).T
summary["chance energy (tumour area share)"] = cam_df["mask_fraction"].mean()
display(Markdown(f"**Grad-CAM vs tumour masks** ({len(cam_df)} Figshare slices, "
                 f"{int(cam_df.correct.sum())} correctly classified)\n\n" + md_table(summary)))
REPORT.append("### E. Grad-CAM vs tumour masks (Figshare)\n\n" + md_table(summary))

# Examples: image with tumour outline, and the heatmap
ex = list(saved_cams)[:6]
fig, axes = plt.subplots(2, len(ex), figsize=(2.4 * len(ex), 5), squeeze=False)
for j, i in enumerate(ex):
    axes[0, j].imshow(fs_img[i], cmap="gray"); axes[0, j].contour(fs_mask[i], levels=[0.5], colors="lime", linewidths=1)
    axes[0, j].set_title(f"true {NAMES[y_fs[i]][:10]}\npred {NAMES[pred_fs[i]][:10]}", fontsize=7)
    axes[1, j].imshow(fs_img[i], cmap="gray"); axes[1, j].imshow(saved_cams[i], cmap="jet", alpha=0.45)
    axes[1, j].contour(fs_mask[i], levels=[0.5], colors="white", linewidths=0.8)
    for a in axes[:, j]:
        a.axis("off")
plt.suptitle("Top: tumour mask (green). Bottom: Grad-CAM with mask outline (white)")
plt.tight_layout(); plt.savefig(FIG_DIR / "gradcam_examples.png", dpi=120); plt.show()

## 3. Save the Phase 1 artefacts and the results summary

Everything is written to `outputs/phase1/` (on Drive when in Colab):

* `ensemble.json`: member models and the ensemble temperature;
* `conformal.json`: the deployed class-conditional thresholds;
* `ood.json` plus `mahalanobis_<seed>.npz`: the deployed OOD detector and its threshold;
* `results_phase1.md`: every table above, ready to paste into the README;
* `figures/`: risk–coverage curve, Figshare confusion matrix, Grad-CAM examples.

In [ ]:
(P1_DIR / "ensemble.json").write_text(json.dumps(
    {"members": [str(f.relative_to(ARTEFACT_DIR)) for f in MODEL_FILES], "temperature": T_ENS}, indent=2))
(P1_DIR / "conformal.json").write_text(json.dumps(CP_DEPLOY.to_dict(), indent=2))
for mh, m in zip(MAHA, members):
    mh.save(P1_DIR / f"mahalanobis_{m['name']}.npz")
(P1_DIR / "ood.json").write_text(json.dumps(
    {"score": "Mahalanobis (mean over members)", "threshold": THRESH["Mahalanobis"], "retain": OOD_RETAIN}, indent=2))
ood_table.to_csv(P1_DIR / "ood_results.csv")
gap.to_csv(P1_DIR / "external_validation.csv")
conf_table.to_csv(P1_DIR / "conformal_results.csv")
ens_table.to_csv(P1_DIR / "ensemble_results.csv")

header = (f"## Phase 1 results\n\nEnsemble of {len(members)} MobileNetV2 models · internal test {len(Y['test'])} images · "
          f"Figshare external {int(shared.sum())} glioma/meningioma slices "
          f"({int(fs_meta['dup_of_training_data'].sum())} duplicates of training-dataset images excluded)\n")
(P1_DIR / "results_phase1.md").write_text(header + "\n" + "\n\n".join(REPORT) + "\n")
print("Saved Phase 1 artefacts to", P1_DIR)

## 4. Inference with an uncertainty-aware pipeline

`predict_image_v2` combines everything:

1. **OOD gate:** if the Mahalanobis score is below the threshold, the image is **rejected** ("doesn't look like the training MRIs") and no tumour type is given.
2. **Ensemble** probabilities, calibrated with the ensemble temperature.
3. A **conformal prediction set** (class-conditional, 90% coverage): every tumour type the model can't rule out.
4. **Member disagreement** (epistemic uncertainty).

> The coverage guarantee only holds for images like the calibration data. Section C shows how it changes on a new dataset.

In [ ]:
def load_rgb(src):
    return ImageOps.exif_transpose(Image.open(src)).convert("RGB")


def predict_image_v2(src, show=True):
    im = load_rgb(src)
    x = tf.round(tf.image.resize(np.asarray(im, dtype=np.float32)[None], IMG_SIZE))
    feats = [m["fmodel"](x, training=False).numpy() for m in members]
    logits = [f @ m["parts"].W + m["parts"].b for f, m in zip(feats, members)]
    maha = float(np.mean([mh.score(f)[0] for mh, f in zip(MAHA, feats)]))
    if show:
        plt.figure(figsize=(3, 3)); plt.imshow(im); plt.axis("off"); plt.show()
    if maha < THRESH["Mahalanobis"]:
        print(f"✋ Rejected: this image doesn't look like the brain MRIs the model was trained on "
              f"(OOD score {maha:.1f} < threshold {THRESH['Mahalanobis']:.1f}). No prediction made.")
        return {"rejected": True, "ood_score": maha}
    mp = np.stack([softmax(l) for l in logits])
    probs = apply_temperature(ensemble_probs(mp), T_ENS)[0]
    pset = CP_DEPLOY.predict_sets(probs[None])[0]
    epi = float(uncertainty(mp)["epistemic"][0])
    for k in np.argsort(-probs):
        print(f"  {NAMES[k]:<34s} {100 * probs[k]:5.1f}%{'   ← in set' if pset[k] else ''}")
    names = [NAMES[k] for k in np.where(pset)[0]]
    if len(names) == 1:
        print(f"→ Prediction set ({100 * (1 - DEPLOY_ALPHA):.0f}% coverage): {names[0]}")
    else:
        print(f"→ Prediction set ({100 * (1 - DEPLOY_ALPHA):.0f}% coverage): {{{', '.join(names)}}}: the model can't "
              f"tell these apart for this image")
    print(f"  member disagreement (epistemic): {epi:.3f} nats")
    print("Research prototype, not a diagnosis. Please discuss MRI findings with a neurologist or neuro-oncologist.")
    return {"rejected": False, "probs": dict(zip(CLASS_NAMES, map(float, probs))),
            "set": [CLASS_NAMES[k] for k in np.where(pset)[0]], "epistemic": epi, "ood_score": maha}


demo_dir = P1_DIR / "demo"
demo_dir.mkdir(exist_ok=True)
print("— internal test image —"); predict_image_v2(PATHS["test"][0], show=False)
if pit.any():
    Image.fromarray(fs_img[np.where(pit)[0][0]]).save(demo_dir / "pituitary.png")
    print("\n— Figshare pituitary slice (unseen tumour type) —"); predict_image_v2(demo_dir / "pituitary.png", show=False)
Image.fromarray(np.clip(rng.normal(128, 50, (128, 128)), 0, 255).astype(np.uint8)).save(demo_dir / "noise.png")
print("\n— noise image —"); _ = predict_image_v2(demo_dir / "noise.png", show=False)

## 5. Limitations of Phase 1

* **External validation covers 2 of 3 classes.** Figshare has no `brain_tumor` class, and the training data has no pituitary class.
* **Figshare images are converted** from raw MRI intensities with a per-slice percentile window. Part of any performance drop may come from this conversion, not from the anatomy or scanner.
* **The duplicate check catches copies, not the same patient.** The training dataset has no patient IDs, so overlap between the two datasets' *patients* can't be ruled out.
* **Conformal guarantees assume exchangeability.** They hold on data like the calibration set; section C measures what happens under dataset shift.
* **OOD sets are examples, not proof.** Passing CIFAR-10 and noise doesn't guarantee rejection of every possible non-MRI input, such as CT or X-ray scans.
* **Grad-CAM is coarse:** a 7×7 map upsampled. Good localisation shows the model *uses* the tumour region, not *how* it decides.
* **No "no tumour" class yet.** This is planned as Phase 1b, with checks that the model can't learn which dataset an image came from.